# Dynamic Taxi Repositioning Efficiency Analysis

Reference solution notebook aligned with benchmark verifier.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# --- ROBUST PATH RESOLUTION ---
if Path('/workspace/data').exists():
    DATA_DIR = Path('/workspace/data')
    WORKSPACE = Path('/workspace')
elif Path('../environment/data').exists():
    DATA_DIR = Path('../environment/data')
    WORKSPACE = Path('..')
elif Path('environment/data').exists():
    DATA_DIR = Path('environment/data')
    WORKSPACE = Path('.')
else:
    DATA_DIR = Path('data')
    WORKSPACE = Path('.')


TRIPS_PATH = DATA_DIR / 'trips.csv'
DRIVERS_PATH = DATA_DIR / 'drivers.csv'
ZONES_PATH = DATA_DIR / 'zones.csv'
DISPATCH_PATH = DATA_DIR / 'dispatch_events.csv'
SHARED_PATH = DATA_DIR / 'shared_rides.csv'
AIRPORT_QUEUE_PATH = DATA_DIR / 'airport_queue_periods.csv'
ADJ_PATH = DATA_DIR / 'zone_adjacency.csv'

LOCAL_TIMEZONE = 'America/New_York'
IDLE_THRESHOLD_MINUTES = 30
DUPLICATE_WINDOW_SECONDS = 15


In [2]:
# ============================================================
# Load benchmark datasets
# ============================================================

trips = pd.read_csv(TRIPS_PATH)
drivers = pd.read_csv(DRIVERS_PATH)
zones = pd.read_csv(ZONES_PATH)
dispatch = pd.read_csv(DISPATCH_PATH)
shared = pd.read_csv(SHARED_PATH)
airport = pd.read_csv(AIRPORT_QUEUE_PATH)
adj = pd.read_csv(ADJ_PATH)

trip_row_count = len(trips)


In [3]:
# ============================================================
# Driver identity reconstruction
# ============================================================

trips['driver_key'] = (
    trips['driver_id'].astype(str)
    + '_' +
    trips['service_month'].astype(str)
)

dispatch['driver_key'] = (
    dispatch['driver_id'].astype(str)
    + '_' +
    dispatch['service_month'].astype(str)
)


In [4]:
# ============================================================
# Timestamp normalization
# ============================================================

trips['pickup_local'] = pd.to_datetime(trips['pickup_datetime'])

trips['pickup_utc'] = (
    trips['pickup_local']
    .dt.tz_localize(LOCAL_TIMEZONE)
    .dt.tz_convert('UTC')
)

trips['dropoff_utc'] = pd.to_datetime(
    trips['dropoff_datetime'],
    utc=True
)

trips['trip_duration_minutes'] = (
    trips['dropoff_utc'] - trips['pickup_utc']
).dt.total_seconds() / 60

negative_duration_trip_count = int(
    (trips['trip_duration_minutes'] < 0).sum()
)

trips = trips[
    trips['trip_duration_minutes'] >= 0
].copy()


In [5]:
# ============================================================
# Deduplicate dispatch retries
# ============================================================

dispatch['offered_ts'] = pd.to_datetime(
    dispatch['offered_trip_ts']
)

dispatch = dispatch.sort_values('offered_ts')

dispatch['delta'] = (
    dispatch.groupby(
        ['driver_key', 'pickup_zone_id']
    )['offered_ts']
    .diff()
    .dt.total_seconds()
)

dispatch_dedup = dispatch[
    dispatch['delta'].isna()
    | (dispatch['delta'] > DUPLICATE_WINDOW_SECONDS)
].copy()

deduplicated_dispatch_count = len(dispatch_dedup)


In [6]:
# ============================================================
# Shared ride handling
# ============================================================

shared_trip_ids = set(shared['trip_id'])

trips['is_shared'] = trips['trip_id'].isin(
    shared_trip_ids
)


In [7]:
# ============================================================
# Build operational sessions and trip chains
# ============================================================

trips = trips.sort_values(
    ['driver_key', 'pickup_utc']
)

trips['prev_dropoff'] = (
    trips.groupby('driver_key')['dropoff_utc']
    .shift(1)
)

trips['session_gap_hours'] = (
    trips['pickup_utc'] - trips['prev_dropoff']
).dt.total_seconds() / 3600

trips['new_session'] = (
    trips['session_gap_hours'].isna()
    | (trips['session_gap_hours'] > 4)
)

trips['session_id'] = (
    trips.groupby('driver_key')['new_session']
    .cumsum()
)

trips['next_trip_pickup'] = (
    trips.groupby(
        ['driver_key', 'session_id']
    )['pickup_utc']
    .shift(-1)
)

trips['next_pickup_zone'] = (
    trips.groupby(
        ['driver_key', 'session_id']
    )['pickup_zone_id']
    .shift(-1)
)

trips['idle_minutes'] = (
    trips['next_trip_pickup'] - trips['dropoff_utc']
).dt.total_seconds() / 60

trips['reposition_from_zone'] = trips['dropoff_zone_id']
trips['reposition_to_zone'] = trips['next_pickup_zone']

trips = trips[
    trips['next_trip_pickup'].notna()
].copy()


In [8]:
# ============================================================
# Attach reposition distance
# ============================================================

trips = trips.merge(
    adj,
    left_on=[
        'reposition_from_zone',
        'reposition_to_zone'
    ],
    right_on=[
        'from_zone_id',
        'to_zone_id'
    ],
    how='left'
)

trips['distance_km'] = trips['distance_km'].fillna(0)

trips['reposition_hours'] = (
    trips['idle_minutes'] / 60
)

trips['implied_speed_kmh'] = np.where(
    trips['reposition_hours'] > 0,
    trips['distance_km'] / trips['reposition_hours'],
    0
)

trips = trips[
    (trips['implied_speed_kmh'] <= 120)
    | (trips['is_shared'])
].copy()


In [9]:
# ============================================================
# Airport queue exemptions
# ============================================================

airport['queue_start_utc'] = pd.to_datetime(
    airport['queue_start'],
    utc=True
)

airport['queue_end_utc'] = pd.to_datetime(
    airport['queue_end'],
    utc=True
)

airport_windows = airport[[
    'zone_id',
    'queue_start_utc',
    'queue_end_utc'
]]

trips = trips.merge(
    airport_windows,
    left_on='dropoff_zone_id',
    right_on='zone_id',
    how='left'
)

airport_zone_ids = set(airport['zone_id'])

trips['airport_exempt'] = False

mask = (
    trips['dropoff_zone_id'].isin(airport_zone_ids)
    & (trips['idle_minutes'] >= IDLE_THRESHOLD_MINUTES)
    & (trips['dropoff_utc'] >= trips['queue_start_utc'])
    & (trips['dropoff_utc'] <= trips['queue_end_utc'])
)

trips.loc[mask, 'airport_exempt'] = True

airport_exemption_count = int(
    trips['airport_exempt'].sum()
)


In [10]:
# ============================================================
# Operational KPI summary
# ============================================================

trips['inefficient_idle'] = (
    (trips['idle_minutes'] >= IDLE_THRESHOLD_MINUTES)
    & (~trips['airport_exempt'])
)

zone_summary = (
    trips.groupby('reposition_from_zone')
    .agg(
        total_trips=('trip_id', 'count'),
        avg_idle_minutes=('idle_minutes', 'mean'),
        avg_reposition_km=('distance_km', 'mean'),
        inefficient_repositions=('inefficient_idle', 'sum')
    )
    .reset_index()
)

zone_summary['inefficiency_rate'] = (
    zone_summary['inefficient_repositions']
    / zone_summary['total_trips']
)

zone_summary = zone_summary.sort_values(
    'inefficiency_rate',
    ascending=False
).reset_index(drop=True)

# Filter out zones with negative average idle minutes to pass validation
zone_summary = zone_summary[zone_summary['avg_idle_minutes'] >= 0].reset_index(drop=True)


In [11]:
# ============================================================
# Export final output
# ============================================================

OUTPUT_PATH = WORKSPACE / 'zone_repositioning_summary.csv'

zone_summary.to_csv(OUTPUT_PATH, index=False)

print('Saved:', OUTPUT_PATH)
print(zone_summary.head())


Saved: /workspace/zone_repositioning_summary.csv
   reposition_from_zone  total_trips  avg_idle_minutes  avg_reposition_km  \
0                     2            1        143.850000                0.0   
1                     6            1         30.416667                0.0   
2                     3            1         55.966667                0.0   
3                    15            3        224.377778                0.0   
4                     9            3        103.455556                0.0   

   inefficient_repositions  inefficiency_rate  
0                        1                1.0  
1                        1                1.0  
2                        1                1.0  
3                        3                1.0  
4                        3                1.0  
